# Step 1: Profiling Numba CUDA - with the Nsight Jupyterlab Extension

In this step, we'll get hands-on experience with Nsight Compute by profiling a Numba CUDA image processing kernel. This kernel implements a common sharpening algorithm. By analyzing the kernel's performance characteristics, we'll identify a bottleneck related to floating-point precision and apply an optimization to achieve significant speedup.

## Numba CUDA

[Numba](https://numba.readthedocs.io/) is a just-in-time (JIT) compiler that translates Python functions to optimized machine code. Its `@cuda.jit` decorator allows writing CUDA kernels directly in Python, giving us the flexibility to implement custom GPU algorithms without being constrained to library calls. While there are many ways to use NVIDIA GPUs from Python (PyTorch, CuPy, etc.), Numba lets us write kernel code that looks quite similar to C++ CUDA.

```python
@numba.cuda.jit
def vector_add(a, b, out):
    idx = numba.cuda.grid(1)
    if idx < out.size:
        out[idx] = a[idx] + b[idx]
```

## The Sharpen Kernel

The kernel we'll profile implements a simple [*unsharp masking*](https://en.wikipedia.org/wiki/Unsharp_masking) algorithm — a common image processing technique used to sharpen images. In this convolution algorithm, a blurred version of the input is combined with the original to produce a sharper output. We don't want to compute the unsharp masking directly in RGB color space, as this can lead to color distortions. Instead, the pixels are first converted to [HSV](https://en.wikipedia.org/wiki/HSL_and_HSV) (hue, saturation, value) format, and the filter is applied only to the value (brightness) component.

Rather than using a simple Gaussian blur, we use a [*bilateral filter*](https://en.wikipedia.org/wiki/Bilateral_filter). This edge-preserving filter combines spatial proximity with intensity similarity, which produces better sharpening results near edges. However, it's more computationally expensive — requiring two `exp()` calls per neighbor instead of one — making it a compute-heavy optimization candidate.

The algorithm works as follows for each pixel:
1. Convert the pixel from RGB to HSV color space
2. Compute a bilateral-filtered blur of the Value (brightness) channel by sampling many neighboring pixels, weighting each by both spatial distance and intensity similarity
3. Apply sharpening: `V_sharp = V_original + (V_original - V_blur) × amount`
4. Convert the result back to RGB

The next cell defines our sharpen kernel. It computes two `exp()` calls per neighbor: one for spatial distance (`spatial_weight`) and one for intensity similarity (`range_weight`).

The kernel is decorated with `lineinfo=True`. This tells Numba to generate line information that allows Nsight Compute to correlate performance metrics with specific lines of source code — invaluable for understanding where time is being spent.

In [ ]:
# run this cell to test our implementation...

import math
import numba
import numba.cuda
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

MAX_RGB = 256.0

@numba.cuda.jit(device=True, lineinfo=True)
def rgb_to_hsv(r, g, b):
    r, g, b = r / MAX_RGB, g / MAX_RGB, b / MAX_RGB
    c_max = max(r, max(g, b))
    c_min = min(r, min(g, b))
    c_diff = c_max - c_min
    if c_diff == 0:
        h = 0.0
    elif c_max == r:
        h = 60.0 * ((g - b) / c_diff)
    elif c_max == g:
        h = 60.0 * (2.0 + (b - r) / c_diff)
    else:
        h = 60.0 * (4.0 + (r - g) / c_diff)
    if h < 0:
        h += 360.0
    s = 0.0 if c_max == 0 else c_diff / c_max
    v = c_max
    return h, s, v

@numba.cuda.jit(device=True, lineinfo=True)
def hsv_to_rgb(h, s, v):
    h_i = math.floor(h / 60.0)
    f = h / 60.0 - h_i
    p = v * (1.0 - s)
    q = v * (1.0 - s * f)
    t = v * (1.0 - s * (1.0 - f))
    if h_i == 0 or h_i == 6:
        r, g, b = v, t, p
    elif h_i == 1:
        r, g, b = q, v, p
    elif h_i == 2:
        r, g, b = p, v, t
    elif h_i == 3:
        r, g, b = p, q, v
    elif h_i == 4:
        r, g, b = t, p, v
    else:
        r, g, b = v, p, q
    return r * MAX_RGB, g, b

@numba.cuda.jit(lineinfo=True)
def sharpen_kernel(img_in, img_out, width, height):
    """Bilateral unsharp masking - uses FP64 by default (Python's float type)."""
    tx, ty = numba.cuda.grid(2)
    x, y = tx, ty
    radius = 8
    if x < radius or x >= width - radius or y < radius or y >= height - radius:
        return

    r, g, b = img_in[y, x, 0], img_in[y, x, 1], img_in[y, x, 2]
    h_orig, s_orig, v_orig = rgb_to_hsv(r, g, b)

    # Bilateral filter parameters
    sigma_s = 4.0   # Spatial sigma
    sigma_r = 0.1   # Range/intensity sigma
    sum_blur = 0.0
    weight_sum = 0.0

    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):
            dist_sq = dx * dx + dy * dy
            if dist_sq <= radius * radius:
                nr, ng, nb = img_in[y + dy, x + dx, 0], img_in[y + dy, x + dx, 1], img_in[y + dy, x + dx, 2]
                _, _, v_neighbor = rgb_to_hsv(nr, ng, nb)
                # Bilateral filter: TWO exp() calls per neighbor
                spatial_weight = math.exp(-dist_sq / (2.0 * sigma_s * sigma_s))
                intensity_diff = v_orig - v_neighbor
                range_weight = math.exp(-intensity_diff * intensity_diff / (2.0 * sigma_r * sigma_r))
                weight = spatial_weight * range_weight
                sum_blur += v_neighbor * weight
                weight_sum += weight

    v_blur = sum_blur / weight_sum
    sharpen_amount = 1.5
    v_sharp = v_orig + (v_orig - v_blur) * sharpen_amount
    v_sharp = max(0.0, min(1.0, v_sharp))

    r_out, g_out, b_out = hsv_to_rgb(h_orig, s_orig, v_sharp)
    img_out[y, x, 0] = r_out
    img_out[y, x, 1] = g_out
    img_out[y, x, 2] = b_out

# Load image from disk
img_pil = Image.open("images/tabby_tiger_cat_blurred.jpg")
img = np.array(img_pil, dtype=np.float32)
print(f"Loaded image: {img.shape}")

# Run kernel
d_img_in = numba.cuda.to_device(img)
d_img_out = numba.cuda.device_array_like(img)
height, width = img.shape[:2]
block = (16, 16)
grid = (math.ceil(width / block[0]), math.ceil(height / block[1]))
sharpen_kernel[grid, block](d_img_in, d_img_out, width, height)
numba.cuda.synchronize()

# Get result
img_out = d_img_out.copy_to_host()
print("Sharpen kernel executed successfully")

# Display input and output
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img.astype(np.uint8))
axes[0].set_title("Input (Blurred)")
axes[0].axis("off")
axes[1].imshow(np.clip(img_out, 0, 255).astype(np.uint8))
axes[1].set_title("Output (Sharpened)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

Looks like the kernel is running but it has a bug. Let's have a quick detour to understand debugging of Numba CUDA kernels with Nvidia's **Nsight Visual Studio Code Edition**. Execute the cell below to create the launch button. After completing the debugging steps, come back here to continue with profiling the fixed kernel.

In [ ]:
# Run this cell to create the debug lab launch button
from IPython.display import HTML, display

# Button that opens a runtime-determined link (hostname from browser at click time)
display(HTML("""
<style>
  .vscode-open-btn {
    font-size: 1.25em;
    font-weight: bold;
    padding: 10px 20px;
    color: white;
    background-color: #0078d4;
    border: none;
    border-radius: 4px;
    cursor: pointer;
  }
  .vscode-open-btn:hover { background-color: #106ebe; }
</style>
<center>
  <button class="vscode-open-btn" type="button"
    onclick="window.open(window.location.protocol + '//' + window.location.host + '/numba-debugging/lab/tree/lab.ipynb', '_blank', 'noopener,noreferrer');">
    Open Debugging Steps
  </button>
</center>
"""))

---

## Profiling with Jupyterlab Extension

<div class="alert alert-block alert-info">
    <p>Exercise: Profile with the Nsight Jupyterlab Extension</p>
</div>

Our environment has the [NVIDIA Nsight Jupyterlab Extension](https://pypi.org/project/jupyterlab-nvidia-nsight/) installed. With it, we can profile individual notebook cells directly from Jupyter and inspect the results in the notebook or in an [Nsight Streamer](https://catalog.ngc.nvidia.com/orgs/nvidia/teams/devtools/containers/nsight-streamer-ncu) based UI in the browser.

To enable profiling:

1. Open the **NVIDIA Nsight** menu in Jupyterlab (top menu bar)
2. Enable **Profiling with Nsight Compute**
3. Make sure that `--set full` is set when prompted for arguments to ncu
4. The notebook kernel will restart under Nsight Compute - **wait for the green run button to be enabled in the toolbar on the top**
5. Select the Python cell below that includes the fixed `sharpen_kernel` and press the green run button (no arguments are needed in this prompt)

<img src="images/jupyterlab_menu.png"/>

<div class="alert alert-block alert-warning">
    <p>Profiling adds significant overhead around the kernel, so the generation will be much slower than normal. This is expected — the profiler is collecting detailed metrics, replaying the kernel multiple times.</p>
</div>
<div class="alert alert-block alert-warning">
    <p>The Open in UI popup button is not setup in our environment.</p>
</div>

In [ ]:
# this is the fixed kernel, use this for profiling
# add any magic commands at the very top, before these comments!
# execute the cell with Shift+Enter after adding the magic command

import math
import numba
import numba.cuda
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

MAX_RGB = 256.0

@numba.cuda.jit(device=True, lineinfo=True)
def rgb_to_hsv(r, g, b):
    r, g, b = r / MAX_RGB, g / MAX_RGB, b / MAX_RGB
    c_max = max(r, max(g, b))
    c_min = min(r, min(g, b))
    c_diff = c_max - c_min
    if c_diff == 0:
        h = 0.0
    elif c_max == r:
        h = 60.0 * ((g - b) / c_diff)
    elif c_max == g:
        h = 60.0 * (2.0 + (b - r) / c_diff)
    else:
        h = 60.0 * (4.0 + (r - g) / c_diff)
    if h < 0:
        h += 360.0
    s = 0.0 if c_max == 0 else c_diff / c_max
    v = c_max
    return h, s, v

@numba.cuda.jit(device=True, lineinfo=True)
def hsv_to_rgb(h, s, v):
    h_i = math.floor(h / 60.0)
    f = h / 60.0 - h_i
    p = v * (1.0 - s)
    q = v * (1.0 - s * f)
    t = v * (1.0 - s * (1.0 - f))
    if h_i == 0 or h_i == 6:
        r, g, b = v, t, p
    elif h_i == 1:
        r, g, b = q, v, p
    elif h_i == 2:
        r, g, b = p, v, t
    elif h_i == 3:
        r, g, b = p, q, v
    elif h_i == 4:
        r, g, b = t, p, v
    else:
        r, g, b = v, p, q
    return r * MAX_RGB, g * MAX_RGB, b * MAX_RGB

@numba.cuda.jit(lineinfo=True)
def sharpen_kernel(img_in, img_out, width, height):
    """Bilateral unsharp masking - uses FP64 by default (Python's float type)."""
    tx, ty = numba.cuda.grid(2)
    x, y = tx, ty
    radius = 8
    if x < radius or x >= width - radius or y < radius or y >= height - radius:
        return

    r, g, b = img_in[y, x, 0], img_in[y, x, 1], img_in[y, x, 2]
    h_orig, s_orig, v_orig = rgb_to_hsv(r, g, b)

    # Bilateral filter parameters
    sigma_s = 4.0   # Spatial sigma
    sigma_r = 0.1   # Range/intensity sigma
    sum_blur = 0.0
    weight_sum = 0.0

    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):
            dist_sq = dx * dx + dy * dy
            if dist_sq <= radius * radius:
                nr, ng, nb = img_in[y + dy, x + dx, 0], img_in[y + dy, x + dx, 1], img_in[y + dy, x + dx, 2]
                _, _, v_neighbor = rgb_to_hsv(nr, ng, nb)
                # Bilateral filter: TWO exp() calls per neighbor
                spatial_weight = math.exp(-dist_sq / (2.0 * sigma_s * sigma_s))
                intensity_diff = v_orig - v_neighbor
                range_weight = math.exp(-intensity_diff * intensity_diff / (2.0 * sigma_r * sigma_r))
                weight = spatial_weight * range_weight
                sum_blur += v_neighbor * weight
                weight_sum += weight

    v_blur = sum_blur / weight_sum
    sharpen_amount = 1.5
    v_sharp = v_orig + (v_orig - v_blur) * sharpen_amount
    v_sharp = max(0.0, min(1.0, v_sharp))

    r_out, g_out, b_out = hsv_to_rgb(h_orig, s_orig, v_sharp)
    img_out[y, x, 0] = r_out
    img_out[y, x, 1] = g_out
    img_out[y, x, 2] = b_out

# Load image from disk
img_pil = Image.open("images/tabby_tiger_cat_blurred.jpg")
img = np.array(img_pil, dtype=np.float32)
print(f"Loaded image: {img.shape}")

# Run kernel
d_img_in = numba.cuda.to_device(img)
d_img_out = numba.cuda.device_array_like(img)
height, width = img.shape[:2]
block = (16, 16)
grid = (math.ceil(width / block[0]), math.ceil(height / block[1]))
sharpen_kernel[grid, block](d_img_in, d_img_out, width, height)
numba.cuda.synchronize()

# Get result
img_out = d_img_out.copy_to_host()
print("Sharpen kernel executed successfully")

# Display input and output
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img.astype(np.uint8))
axes[0].set_title("Input (Blurred)")
axes[0].axis("off")
axes[1].imshow(np.clip(img_out, 0, 255).astype(np.uint8))
axes[1].set_title("Output (Sharpened)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In our setup, there are two limitations related to this extension:

- Opening the UI from the popup directly would allow us to see only a single report at a time.
- When profiling directly from a Jupyter notebook cell, ncu cannot access and import the high-level Python source directly (vs. when profiling a standalone script). The current version of the extension doesn't support mapping exported versions of the cell into its container for resolving the source view.

To workaround this:
- Start a new Nsight Compute UI by selecting **NVIDIA Nsight** > **Display Nsight UI via Nsight Streamer** from the Jupyterlab menu.
- Navigate to `/home/nvidia/reports/` in the UI and open the `step...ncu-rep` new report file.

<img src="images/jupyterlab_menu_2.png"/>

## Identifying the Bottleneck

Whenever possible, Nsight Compute shows a list of **prioritized rules** on the Summary and Details pages. These rules are actually [implemented in Python](https://docs.nvidia.com/nsight-compute/CustomizationGuide/index.html#rule-system) and analyze the collected metrics to suggest optimizations. The ordering is based on the estimated speedup that the kernel may achieve by fully implementing the rule's suggestion. Of course, these estimates should be considered upper bounds — resolving the detected problem won't always achieve the promised speedup, but they provide excellent guidance on where to focus optimization efforts.

Together, we inspect the report in the UI and investigate:

1. **Summary Page** — Check the *Prioritized Rules* list. What's the top optimization suggestion? Click the rule's link to jump directly to the associated section on the Details page.
2. **Details Page > GPU Speed of Light** — Is the kernel compute-bound or memory-bound? Look at the relative utilization of compute (SM) vs memory.
3. **Details Page > Compute Workload Analysis** — Which pipelines have high utilization? This section breaks down which specific hardware units are being used.

**Expected findings** (expand after trying the exercise):

- The **Summary Page** shows *FP64/32 Utilization* as an important rule, suggesting the kernel could be faster if it used FP32 instead of FP64. The estimated speedup gives us a target to aim for.
- **GPU Speed of Light** shows high SM (compute) utilization but low memory utilization → the kernel is **compute-bound**, meaning we're limited by SM throughput, not memory bandwidth. This is important because different bottlenecks require different optimization strategies.
- **GPU Speed of Light Breakdown** shows **SM Issue Active** as the main bottleneck. The SM can't find enough instructions to issue each cycle, typically due to high latency instructions, limited ILP, low occupancy
- **Compute Workload Analysis** shows **Pipe FP64** utilization while FP32 is barely used. This confirms the rule's finding — our kernel is doing lots of double-precision floating-point operations.

**Why does this happen?** 
- Python's default float type is 64-bit (double-precision). Every literal like `2.5`, `60.0`, `256.0` and every `math.exp()` call in our kernel uses FP64 arithmetic. On H100, FP64 throughput is only half of FP32 throughput, so by using double-precision when we don't need it, we're leaving performance on the table. For an image processing kernel working with 8-bit color values, 32-bit precision is more than sufficient.
- For Transcendental Functions like `exp`, FP64 has higher latency than FP32, since it can't use the Special Function Unit (SFU)

---

## Resolving the Source

When profiling directly from a Jupyter notebook cell, ncu cannot access and import the high-level Python source directly (vs. when profiling a standalone script).
To still allow us to understand per-line metrics, we can resolve it from a file afterwards:

1. Add the above **magic command** to the start of the notebook cell we just profiled
2. Run the cell (without profiling) to have its contents written to disk
3. Switch to the Nsight Compute UI tab, and on the Source page, select the **Resolve button**
4. Select `All Files (*)` as the type, and then `/home/nvidia/source/sharpen1.py`

With the Source page resolved, we can

- use the *FP64/FP32 Utilization* table on the Details page to quickly find the lines with the most FP64 instructions
- use the SASS view's new *Metric Pipelines* column to find instructions mapping to the *FP64* pipeline
- use the *Instruction Mix* column on the CUDA source view to find lines with many *Floating Point* instructions

--- 

## Optimization

Now that we've identified the bottleneck using the profiler, let's fix it. The solution is straightforward: we need to tell Numba to use 32-bit (single-precision) floats instead of Python's default 64-bit. This is a common pattern when porting Python code to GPU — Python's convenience of automatic typing works against performance when double-precision isn't needed.

To accomplish this, we wrap all floating-point values with `numba.float32()`. This explicit cast tells Numba to use 32-bit precision for that value and all subsequent operations involving it.

Key changes to our code:
- All float literals: `4.0` → `numba.float32(4.0)`
- Constants: `MAX_RGB = 256.0` → `MAX_RGB = numba.float32(256.0)`
- Transcendental functions (`math.exp()`, `math.log()`) automatically use FP32 when their inputs are FP32

While this makes the code more verbose, it gives us explicit control over precision — and as we'll see, the performance improvement is substantial.

Profile the optimized FP32 version by running it under Nsight Jupyterlab Extension:

In [ ]:
import math
import numba
import numba.cuda
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# FP32 constant instead of FP64
MAX_RGB = numba.float32(256.0)

@numba.cuda.jit(device=True, lineinfo=True)
def rgb_to_hsv(r, g, b):
    r, g, b = r / MAX_RGB, g / MAX_RGB, b / MAX_RGB
    c_max = max(r, max(g, b))
    c_min = min(r, min(g, b))
    c_diff = c_max - c_min
    if c_diff == 0:
        h = numba.float32(0.0)
    elif c_max == r:
        h = numba.float32(60.0) * ((g - b) / c_diff)
    elif c_max == g:
        h = numba.float32(60.0) * (numba.float32(2.0) + (b - r) / c_diff)
    else:
        h = numba.float32(60.0) * (numba.float32(4.0) + (r - g) / c_diff)
    if h < 0:
        h += numba.float32(360.0)
    s = numba.float32(0.0) if c_max == 0 else c_diff / c_max
    v = c_max
    return h, s, v

@numba.cuda.jit(device=True, lineinfo=True)
def hsv_to_rgb(h, s, v):
    h_i = math.floor(h / numba.float32(60.0))
    f = h / numba.float32(60.0) - h_i
    p = v * (numba.float32(1.0) - s)
    q = v * (numba.float32(1.0) - s * f)
    t = v * (numba.float32(1.0) - s * (numba.float32(1.0) - f))
    if h_i == 0 or h_i == 6:
        r, g, b = v, t, p
    elif h_i == 1:
        r, g, b = q, v, p
    elif h_i == 2:
        r, g, b = p, v, t
    elif h_i == 3:
        r, g, b = p, q, v
    elif h_i == 4:
        r, g, b = t, p, v
    else:
        r, g, b = v, p, q
    return r * MAX_RGB, g * MAX_RGB, b * MAX_RGB

@numba.cuda.jit(lineinfo=True)
def sharpen_kernel(img_in, img_out, width, height):
    """Bilateral unsharp masking - now with FP32."""
    tx, ty = numba.cuda.grid(2)
    x, y = tx, ty
    radius = 8
    if x < radius or x >= width - radius or y < radius or y >= height - radius:
        return

    r, g, b = img_in[y, x, 0], img_in[y, x, 1], img_in[y, x, 2]
    h_orig, s_orig, v_orig = rgb_to_hsv(r, g, b)

    # FP32 bilateral filter parameters
    sigma_s = numba.float32(4.0)   # Spatial sigma
    sigma_r = numba.float32(0.1)   # Range/intensity sigma
    sum_blur = numba.float32(0.0)
    weight_sum = numba.float32(0.0)

    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):
            dist_sq = numba.float32(dx * dx + dy * dy)
            if dist_sq <= radius * radius:
                nr, ng, nb = img_in[y + dy, x + dx, 0], img_in[y + dy, x + dx, 1], img_in[y + dy, x + dx, 2]
                _, _, v_neighbor = rgb_to_hsv(nr, ng, nb)
                # Bilateral filter: TWO exp() calls per neighbor - FP32
                spatial_weight = math.exp(-dist_sq / (numba.float32(2.0) * sigma_s * sigma_s))
                intensity_diff = v_orig - v_neighbor
                range_weight = math.exp(-intensity_diff * intensity_diff / (numba.float32(2.0) * sigma_r * sigma_r))
                weight = spatial_weight * range_weight
                sum_blur += v_neighbor * weight
                weight_sum += weight

    v_blur = sum_blur / weight_sum
    sharpen_amount = numba.float32(1.5)
    v_sharp = v_orig + (v_orig - v_blur) * sharpen_amount
    v_sharp = max(numba.float32(0.0), min(numba.float32(1.0), v_sharp))

    r_out, g_out, b_out = hsv_to_rgb(h_orig, s_orig, v_sharp)
    img_out[y, x, 0] = r_out
    img_out[y, x, 1] = g_out
    img_out[y, x, 2] = b_out

# Load image from disk
img_pil = Image.open("images/tabby_tiger_cat_blurred.jpg")
img = np.array(img_pil, dtype=np.float32)
print(f"Loaded image: {img.shape}")

# Run kernel
d_img_in = numba.cuda.to_device(img)
d_img_out = numba.cuda.device_array_like(img)
height, width = img.shape[:2]
block = (16, 16)
grid = (math.ceil(width / block[0]), math.ceil(height / block[1]))
sharpen_kernel[grid, block](d_img_in, d_img_out, width, height)
numba.cuda.synchronize()

# Get result
img_out = d_img_out.copy_to_host()
print("Sharpen FP32 kernel executed successfully")

# Display input and output
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img.astype(np.uint8))
axes[0].set_title("Input (Blurred)")
axes[0].axis("off")
axes[1].imshow(np.clip(img_out, 0, 255).astype(np.uint8))
axes[1].set_title("Output (Sharpened - FP32)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

---

## Comparing Results

<div class="alert alert-block alert-warning">
    <p>Disable the Nsight Jupyterlab extension before executing the cell that uses the Python Report Interface.</p>
</div>

<img src="images/jupyterlab_menu.png"/>

**What to look for:**
- **Duration** — How much faster is the FP32 version? This is the bottom-line metric that tells you if your optimization worked.
- **Compute Workload Analysis** — FP64 pipe utilization should drop to ~0%, confirming we eliminated the bottleneck
- **FMA Pipe** — Should now show higher utilization, indicating we're using the faster hardware path

On H100, FP64 throughput is half of FP32 throughput (on other chips, it's as low as 1:32). Switching to FP32 should give us up to **~2× speedup**. The baseline comparison will show exactly how much we gained, and we can verify that the improvement matches the profiler's original estimate.

To save some time, we prepared a **Python Report Interface script** that quickly compares the key metrics from our two reports. Execute the cell below to check the results:

In [ ]:
!python3 source/compare_reports.py

When doing this training offline, feel free to compare results in the UI using the Baselines feature:

### Comparing with Baselines

One of Nsight Compute's most powerful features is the ability to compare results using baselines. This makes it easy to quantify the impact of optimizations and verify that changes had the expected effect. When a baseline is set, all metric values in the current report are shown relative to the baseline.

<div class="alert alert-block alert-info">
    <p>Exercise: Compare Original vs Optimized</p>
</div>

In the Nsight Compute UI:

1. With the **first report** open/focused, click **Compare > Add Baseline** to set it as the baseline for comparison
3. Switch to the optimized report
4. All metrics and charts now show relative differences — you can immediately see where a valu increased or decreased

<img src="images/add_baseline.png"/>

### Summary

<div class="alert alert-block alert-success">
    <p><b>ncu CLI</b>: Use <code>--set full</code> for comprehensive metrics.</p>
    <p><b>UI Navigation</b>: Summary page for prioritized rules, Details page for metrics, Source page for per-line analysis.</p>
    <p><b>Source import</b>: You can resolve source of collected reports from local files.</p>
    <p><b>FP64 Bottleneck</b>: Python defaults to 64-bit floats. Check Compute Workload Analysis for Pipe FP64 utilization.</p>
    <p><b>FP32 Optimization</b>: Use <code>numba.float32()</code> casts to force 32-bit precision. ~2× speedup on H100.</p>
</div>

**Next: [Step 2 - Profiling LLM Inference](step2.ipynb)**